# Detect Auto-Label Harvest — Preds as Editable GT

**Purpose:** generate YOLO-detect labels from model preds so you only correct misses. Mirrors `auto_label_pose.ipynb`.

1. Edit `TARGET_CLASSES` in the next cell — add entries as you add classes (IDs auto-derive from order; never reorder, only append).
2. Edit `AUTO_LABEL_SOURCES` — add `(video, ranges)` entries (or set `AUTO_LABEL_ALL_VIDEOS = True` to harvest every `*.mp4` in `video/input/` wholesale; explicit entries take precedence, `VIDEO_SKIP_STEMS` excludes videos).
3. Run the Runner cell → it writes `datasets/auto_label_detect_v1` (`images/train/*.jpg` + `labels/train/*.txt` + `data.yaml`).
4. `zip datasets/auto_label_detect_v1` → upload to your Platform dataset → correct in editor.

**Models:** `BEST_MODEL` (`models/detect/best.pt`, your trained 6-class weights) pre-labels every class in `TARGET_CLASSES` via name lookup + `BEST_ALIASES` (`cell phone` → `phone`). A class with no match prints a warning and is left manual-only (empty pre-labels).

**Filenames are video-prefixed:** `<mp4stem>_<ms7d>.jpg`, e.g. `vid09_0115000.jpg` — same convention as pose, so frames from different videos never collide.

Leave `AUTO_LABEL_SOURCES = []` to skip safely. `PRE_LABEL_CONF=0.20` surfaces weak boxes for review (same as hoid runtime).

In [1]:
### Auto-label harvest (detect) — CONFIG (generic, edit classes + sources below)
from pathlib import Path
import numpy as np
import cv2
from ultralytics import YOLO as _YOLO2


### Absolute paths: resolve once against wherever this kernel launched.
### BASE = the directory holding models/ (i.e. src/). No chdir anywhere.
_BASE_CANDS = [Path.cwd(), Path.cwd() / 'src', Path.cwd().parent]
BASE = next((p for p in _BASE_CANDS if (p / 'models' / 'detect' / 'best.pt').exists()), None)
if BASE is None:
    raise FileNotFoundError('best.pt not found: launch this notebook from thesis root, src/, or src/labelers/')
print('BASE:', BASE, flush=True)

AUTO_LABEL_SOURCES = [
    ("video/input/Scenarios/DrinkBottle.mp4", None),
    ("video/input/Scenarios/DrinkCup.mp4", None),
    ("video/input/Scenarios/LeaveObjectTaken.mp4", None),
    ("video/input/Scenarios/MultipleObjectUnattended.mp4", None),
    ("video/input/Scenarios/ObjectOccluded.mp4", None),
    ("video/input/Scenarios/ObjectPassed.mp4", None),
    ("video/input/Scenarios/PickingUpPhone.mp4", None),
    ("video/input/Scenarios/PickUpBag.mp4", None),
    ("video/input/Scenarios/ReachingNotTouching.mp4", None),
    ("video/input/Scenarios/UseWallet.mp4", None),
    ("video/input/Scenarios/WorkLaptop.mp4", None),
    ("video/input/Angle3.mp4", None),

    ("video/input/vid09.mp4", None),
    ("video/input/vid12.mp4", [(259, 274)]),
    ("video/input/vid06.mp4", [(0, 4), (10, 12), (19, 24), (102, 111)]),
    ("video/input/vid05.mp4", [(31, 60)]),
    ("video/input/vid20.mp4", None),
    ("video/input/vid34.mp4", None),
    ("video/input/Angle1.mp4", [(287, 297), (450, 460), (469, 476), (929, 939)]),
    ("video/input/vid23.mp4", [(12, 22), (191, 212)]),
    ("video/input/vid13.mp4", [(255, 265)]),
    ("video/input/vid22.mp4", [(92, 140), (211, 221)]),
    ("video/input/vid16.mp4", [(33, 39)]),
    ("video/input/vid26.mp4", [(5, 20)]),
    ("video/input/Angle2.mp4", None),
]
AUTO_LABEL_ALL_VIDEOS = False ### when True, harvest EVERY *.mp4 in VIDEO_INPUT_DIR (whole video each)
VIDEO_INPUT_DIR = BASE / "video" / "input"  ### source folder globbed when AUTO_LABEL_ALL_VIDEOS is True
VIDEO_SKIP_STEMS = set()  ### e.g. {"vid04"} — video stems to skip when globbing
FRAME_EVERY_S  = 2        ### sampling density in seconds
PRE_LABEL_CONF = 0.20     ### LOW-ish: weak boxes surface for your review (hoid runtime uses 0.2)
AUTO_OUT       = BASE / "datasets" / "auto_label_detect"
BEST_MODEL = str(BASE / "models" / "detect" / "best.pt")  ### single multi-class model for EVERY target class
BEST_ALIASES = {"cell phone": "phone"}  ### model name -> TARGET_CLASSES name (exact match otherwise)
IMGSZ          = 640
DUP_IOU        = 0.5       ### per-class NMS: drop boxes overlapping > this IoU with a higher-conf box (same class, same frame)

TARGET_CLASSES = ["phone", "laptop", "wallet", "bottle", "bag", "cup"]
### IDs auto-derive from order — never reorder, only append (or old label IDs shift).
### e.g. TARGET_CLASSES = ["phone", "laptop", "tablet"]  ->  {phone: 0, laptop: 1, tablet: 2}
CLASS_MAP = {name: i for i, name in enumerate(TARGET_CLASSES)}

### Single-model setup: every TARGET_CLASSES entry must match a best.pt class name
### after BEST_ALIASES; unmatched classes print a warning and stay manual-only.

assert len(set(TARGET_CLASSES)) == len(TARGET_CLASSES), "TARGET_CLASSES must be unique"

def _nms(xyxy, scores, iou_thr=DUP_IOU):
    """Greedy NMS: keep highest-conf box, drop boxes with IoU > iou_thr. Returns kept positions (0..n-1)."""
    xyxy = np.asarray(xyxy, dtype=np.float64)
    scores = np.asarray(scores, dtype=np.float64)
    if xyxy.size == 0 or scores.size == 0:
        return []
    order = np.argsort(-scores)
    keep = []
    while order.size:
        i = order[0]; keep.append(int(i))
        if order.size == 1: break
        rest = order[1:]
        xx1 = np.maximum(xyxy[i,0], xyxy[rest,0]); yy1 = np.maximum(xyxy[i,1], xyxy[rest,1])
        xx2 = np.minimum(xyxy[i,2], xyxy[rest,2]); yy2 = np.minimum(xyxy[i,3], xyxy[rest,3])
        inter = np.maximum(0., xx2-xx1) * np.maximum(0., yy2-yy1)
        area_i = (xyxy[i,2]-xyxy[i,0])*(xyxy[i,3]-xyxy[i,1])
        area_r = (xyxy[rest,2]-xyxy[rest,0])*(xyxy[rest,3]-xyxy[rest,1])
        iou = inter / (area_i + area_r - inter + 1e-9)
        order = rest[np.flatnonzero(iou <= iou_thr)]
    return keep


BASE: c:\Users\Cent\Desktop\thesisRun\thesis\src


In [2]:
import shutil
# shutil.rmtree(AUTO_OUT)  ### uncomment to wipe the previous harvest before re-running

In [3]:
from collections import defaultdict

_best = _YOLO2(BEST_MODEL)
_mnames_lc = {int(k): str(v).lower() for k, v in _best.names.items()}
_mname_to_idx = {v: k for k, v in _mnames_lc.items()}
_talias_rev = {v: k for k, v in BEST_ALIASES.items()}  ### TARGET_CLASSES name -> model name
_best_idx_to_target = {}  ### model cls idx -> (target_name, target_id)
for _c in TARGET_CLASSES:
    _lookup = str(_talias_rev.get(_c, _c)).lower()
    if _lookup in _mname_to_idx:
        _ci = _mname_to_idx[_lookup]
        _best_idx_to_target[_ci] = (_c, CLASS_MAP[_c])
    else:
        print(f"[auto_label_detect] WARNING: '{_c}' not in best.pt names — manual-only (no pre-labels).")
print(f"[auto_label_detect] best.pt mapping: {_best_idx_to_target} | model_names={_mnames_lc}")

_img_dir = AUTO_OUT / "images" / "train"
_lbl_dir = AUTO_OUT / "labels" / "train"
_img_dir.mkdir(parents=True, exist_ok=True); _lbl_dir.mkdir(parents=True, exist_ok=True)
_written = 0
_per_class = defaultdict(int)

_sources = list(AUTO_LABEL_SOURCES)
if AUTO_LABEL_ALL_VIDEOS:
    _have = {Path(_v).stem for _v, _r in _sources}  ### explicit entries win; glob skips them (same stem+ms names would overwrite)
    for _p in sorted(Path(VIDEO_INPUT_DIR).glob("*.mp4")):
        if _p.stem in VIDEO_SKIP_STEMS or _p.stem in _have:
            continue
        _sources.append((str(_p), None))
        _have.add(_p.stem)
print(f"[auto_label_detect] sources ({len(_sources)}): {[Path(_v).stem for _v, _r in _sources]}")

for _vid, _rngs in _sources:
    _cap = cv2.VideoCapture(str(_vid) if Path(_vid).is_absolute() else str(BASE / _vid))
    assert _cap.isOpened(), _vid
    _fps = _cap.get(cv2.CAP_PROP_FPS) or 25.0
    _total = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    _spans = _rngs or [(0, _total / _fps)]
    _stem = Path(_vid).stem  ### e.g. "vid09" from "video/input/vid09.mp4"
    for _s0, _s1 in _spans:
        _t = _s0
        while _t <= min(_s1, _total / _fps):
            _cap.set(cv2.CAP_PROP_POS_MSEC, _t * 1000)
            _ok, _fr = _cap.read()
            if not _ok: break
            ### Video-prefixed name: <mp4stem>_<ms> — unique across videos even at the same timestamp
            _name = f"{_stem}_{int(_t * 1000):07d}"
            cv2.imwrite(str(_img_dir / f"{_name}.jpg"), _fr)
            _boxes = []  ### (target_id, x1, y1, x2, y2, conf)
            _r = _best.predict(_fr, conf=PRE_LABEL_CONF, imgsz=IMGSZ, verbose=False)[0]
            if _r.boxes is not None and len(_r.boxes):
                _xy = _r.boxes.xyxy.cpu().numpy(); _cf = _r.boxes.conf.cpu().numpy()
                _cl = _r.boxes.cls.cpu().numpy().astype(int)
                for _b, _s, _c in zip(_xy, _cf, _cl):
                    _hit = _best_idx_to_target.get(int(_c))
                    if _hit is None: continue
                    _tname, _tid = _hit
                    _boxes.append((_tid, float(_b[0]), float(_b[1]), float(_b[2]), float(_b[3]), float(_s)))
            ### Per-class NMS (a phone overlapping a laptop must NOT suppress it)
            _grouped = defaultdict(list)
            for _i, _b in enumerate(_boxes):
                _grouped[_b[0]].append(_i)
            _lines = []
            _W, _H = _fr.shape[1], _fr.shape[0]
            for _tid, _idxs in _grouped.items():
                _gxy = np.array([[_boxes[i][1], _boxes[i][2], _boxes[i][3], _boxes[i][4]] for i in _idxs])
                _gsc = np.array([_boxes[i][5] for i in _idxs])
                for _k in _nms(_gxy, _gsc):
                    _i = _idxs[_k]
                    _, _x1, _y1, _x2, _y2, _ = _boxes[_i]
                    _cx, _cy = (_x1 + _x2) / 2 / _W, (_y1 + _y2) / 2 / _H
                    _bw, _bh = (_x2 - _x1) / _W, (_y2 - _y1) / _H
                    _lines.append(f"{_tid} {_cx:.6f} {_cy:.6f} {_bw:.6f} {_bh:.6f}")
                    _per_class[_tid] += 1
            (_lbl_dir / f"{_name}.txt").write_text("\n".join(_lines))  ### empty file if no preds — keeps image/label parity
            _written += 1; _t += FRAME_EVERY_S
    _cap.release()

_names_lines = "\n".join(f"  {CLASS_MAP[c]}: {c}" for c in sorted(TARGET_CLASSES, key=lambda c: CLASS_MAP[c]))
(AUTO_OUT / "data.yaml").write_text(
    f"path: {AUTO_OUT.resolve().as_posix()}\ntrain: images/train\nval: images/train\nnames:\n{_names_lines}\n"
)
print(f"[auto_label_detect] frames={_written} per_class={dict(_per_class)} names={_names_lines.replace(chr(10), ' | ')}")
zip_path = shutil.make_archive(str(AUTO_OUT), 'zip', str(AUTO_OUT))
print(f"[auto_label_detect] zipped -> {zip_path}")

[auto_label_detect] best.pt mapping: {0: ('phone', 0), 1: ('laptop', 1), 2: ('wallet', 2), 3: ('bottle', 3), 4: ('bag', 4), 5: ('cup', 5)} | model_names={0: 'cell phone', 1: 'laptop', 2: 'wallet', 3: 'bottle', 4: 'bag', 5: 'cup'}
[auto_label_detect] sources (25): ['DrinkBottle', 'DrinkCup', 'LeaveObjectTaken', 'MultipleObjectUnattended', 'ObjectOccluded', 'ObjectPassed', 'PickingUpPhone', 'PickUpBag', 'ReachingNotTouching', 'UseWallet', 'WorkLaptop', 'Angle3', 'vid09', 'vid12', 'vid06', 'vid05', 'vid20', 'vid34', 'Angle1', 'vid23', 'vid13', 'vid22', 'vid16', 'vid26', 'Angle2']
[auto_label_detect] frames=3163 per_class={5: 7522, 0: 2328, 2: 4248, 4: 1541, 1: 1861, 3: 1369} names=  0: phone |   1: laptop |   2: wallet |   3: bottle |   4: bag |   5: cup
[auto_label_detect] zipped -> c:\Users\Cent\Desktop\thesisRun\thesis\src\datasets\auto_label_detect.zip
